# POC using Guidance

In [ ]:
from __future__ import annotations

import os

import guidance as gi
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
llama3_model = AutoModelForCausalLM.from_pretrained(
    "../model/Llama-3.2-1B-Instruct", trust_remote_code=True
)
llama3_tokenizer = AutoTokenizer.from_pretrained(
    "../model/Llama-3.2-1B-Instruct", trust_remote_code=True
)

llama3_model.to("cuda")
llama3_model.eval()

In [ ]:
llama31_template = llama3_tokenizer.chat_template


class Llama3ChatTemplate(gi.chat.ChatTemplate):
    # available_roles = ["user", "assistant"]
    template_str = llama31_template

    def get_role_start(self, role_name):
        if role_name == "system":
            return "<|start_header_id|>system<|end_header_id|>\n\nYou are helpful!"
        if role_name == "user":
            return "<|start_header_id|>user<|end_header_id|>\n\n"
        if role_name == "assistant":
            return "<|start_header_id|>assistant<|end_header_id|>\n\n"
        raise gi.chat.UnsupportedRoleException(role_name, self)

    def get_role_end(self, role_name=None):
        del role_name  # unused needed for compatibility
        return "<|eot_id|>"

In [ ]:
llama3 = gi.models.Transformers(
    model=llama3_model,
    tokenizer=llama3_tokenizer,
    chat_template=Llama3ChatTemplate,
    ignore_bos_token=True,
)

In [ ]:
# Define multiple questions
questions = [
    "What is the capital of France?",
    "Who discovered gravity?",
    "What is the boiling point of water in Celsius?",
    "Who wrote '1984'?",
]
# Process each question independently while keeping the model loaded
for question in questions:
    lm = llama3.copy()  # Create a new independent model instance
    with gi.system():
        lm += "You are a helpful assistant."
    with gi.user():
        lm += question
    with gi.assistant():
        lm += "\nAnswer: " + gi.select(
            ["Belgium", "Whale", "Train", "Unknown"], name="Answer"
        )
    with gi.user():
        lm += f"Why is {lm['Answer']} correct?"
    with gi.assistant():
        lm += gi.gen(name="Reason", stop="\n\n", max_tokens=200)

    print(f"{lm}\n")

    print("+++ Answer:", lm["Answer"], "\n\n")

In [ ]:
llama3_tokenizer.apply_chat_template(
    [
        {"role": "user", "content": "Hello, how are you?"},
        {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
    ],
    tokenize=False,
)

In [ ]:
lm = llama3.copy()  # Create a new independent model instance (shares GPU resources)

sentence = "Hello how are you? None"

with gi.user():
    lm += "Hello, can you please give me some information about France in a JSON? "
    lm += "Can this include the keys 'capital' and the 'population'?"
with gi.assistant():
    lm += gi.json(name="output", max_tokens=400)

lm["output"]

In [ ]:
lm = llama3.copy()  # Create a new independent model instance (shares GPU resources)

sentence = "Hello how are you? None Apple Rock"

with gi.user():
    lm += "Name a fruit."
with gi.assistant():
    lm += "A fruit is a " + gi.substring(sentence, name="output") + "!"

lm["output"]